# 05 - Visualizzazione Territoriale della Capitanata

In [4]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'test-download-capitanata'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Colab rilevato. Inizializzazione in corso...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 5.72 KiB | 1.91 MiB/s, done.
From https://github.com/SimoRinaldi/crop-spatial-classification
   7a73cc9..3080908  test-download-capitanata -> origin/test-download-capitanata
branch 'test-download-capitanata' set up to track 'origin/test-download-capitanata'.
Reset branch 'test-download-capitanata'
Your branch is up to date with 'origin/test-download-capitanata'.
HEAD is now at 3080908 test predizione capitanata
Setup ambiente Colab completato! Branch attivo su Colab:  test-download-capitanata
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/MyDrive/Progetto MLDM/data/raw


In [ ]:
import json
import pandas as pd
from pathlib import Path

# caricamento del dataset e della legenda
dataset_path = PROCESSED_DIR / "dataset" / "dataset.parquet"
# points_path = INTERIM_DIR / "points.json"
legend_path = PROCESSED_DIR / "legend.json"

df_dataset = pd.read_parquet(dataset_path)

with open(legend_path, "r", encoding="utf-8") as f:
    legend = json.load(f)

print(f"Dataset caricato con {len(df_dataset)} campi.")
print(f"Legenda disponibile con {len(legend)} classi colturali.")

Dataset caricato con 3137 campi.
Legenda disponibile con 21 classi colturali.


#### Predizione dell'intera Capitanata
flattening dell'immagine -> predizione -> reshape mappa

In [ ]:
import joblib
from src.feature_engineering import (                        
    FEATURE_LIST,                                           
    load_monthly_capitanata_rasters,                         
    extract_features_from_monthly_stack                      
)                              

# caricamento del modello
MODELS_DIR = DATA_DIR / "models"
model_path = MODELS_DIR / "random_forest_crop_model.joblib"
model = joblib.load(model_path)
print("Modello caricato!")

# caricamento dei 12 TIF mensili della Capitanata                          
capitanata_dir = PROCESSED_DIR / "sentinel2_capitanata_area" 
monthly_stack, profile = load_monthly_capitanata_rasters(capitanata_dir, year=2023, max_size=1000)                                                 
                                                                
n_months, n_bands, h, w = monthly_stack.shape                
print(f"Stack mensile caricato: {n_months} mesi, {n_bands} bande, risoluzione: {h}x{w} pixel.")                           
                                                                
# estrazione vettoriale delle feature                                                   
_, features_matrix = extract_features_from_monthly_stack(monthly_stack)             
print(f"Matrice feature generata: {features_matrix.shape[0]:,} pixel x {features_matrix.shape[1]} feature.")                        
                                                                
# predizione spaziale su tutti i pixel                    
pred_flat = model.predict(features_matrix)                         
pred_map = pred_flat.reshape(h, w)                           
print("Predizione completata!")  

Modello caricato!
✅ Stack mensile caricato: 12 mesi, 6 bande, risoluzione: 965x1000 pixel.


/content/crop-spatial-classification/src/feature_engineering.py:183: RuntimeWarning: divide by zero encountered in divide
  features_dict["Orzo_Wheat_Ratio"] = ndvi_series[3] / (ndvi_series[4] + 0.01) # Aprile / Maggio


✅ Matrice feature generata: 965,000 pixel x 51 feature.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch   

# caricamento di un mese della Capitanata e conversione in immagine RGB
image = monthly_stack[5] # Giugno
# Rasterio ordina 1-based: [1:Blu, 2:Verde, 3:Rosso, ...]
# NumPy ordina 0-based: [0:Blu, 1:Verde, 2:Rosso, ...]
# [2, 1, 0] per RGB
rgb_img = np.dstack((image[2], image[1], image[0]))

# normalizzazione per visualizzazione
rgb_img = rgb_img / np.percentile(rgb_img, 98)
rgb_img = np.clip(rgb_img, 0, 1)

# -------------------------------------------------------------------
# recupero dei colori ufficiali Copernicus                                       
clr_files = list(RAW_DIR.rglob("*.clr"))                     
color_map = {}                                               
                               
if clr_files:                                                
    with open(clr_files[0], "r", encoding="utf-8") as f:     
        for line in f:                                       
            parts = line.strip().split()                     
            if len(parts) >= 4:                              
                code = int(parts[0])                         
                # normalizzazione dei valori RGB tra 0.0 e 1.0 per matplotlib                                                     
                color_map[code] = [
                    int(parts[1]) / 255.0, 
                    int(parts[2]) / 255.0, 
                    int(parts[3]) / 255.0
                ]

# colorazione della mappa predetta
h, w = pred_map.shape
pred_rgb = np.ones((h, w, 3), dtype=np.float32) * 0.95 # sfondo grigio chiaro per pixel non classificati

for code, color in color_map.items():
    mask = (pred_map == code)
    if np.any(mask):
        pred_rgb[mask] = color

# -------------------------------------------------------------------
# creazione del plot con due grafici affiancati
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# immagine satellitare RGB
axes[0].imshow(rgb_img)
axes[0].set_title('Immagine reale Capitanata (RGB)')
axes[0].axis('off')

# predizione rgb
axes[1].imshow(pred_rgb)
axes[1].set_title('Predizione della Capitanata (Random Forest)')
axes[1].axis('off')

# legenda dinamica (mostra solo le classi predette)                                     
unique_predictions = sorted(np.unique(pred_map))                   
legend_handles = [                                           
    Patch(                                                   
        facecolor=color_map.get(int(c), [0.5, 0.5, 0.5]),    
        edgecolor="black",                                   
        label=legend.get(str(int(c)), f"Classe {c}")         
    )                                                        
    for c in unique_predictions if int(c) in color_map and int(c) != 0                                                           
]                                                            
                                                                
axes[1].legend(                                              
    handles=legend_handles,                                  
    loc="center left",                                       
    bbox_to_anchor=(1.02, 0.5),                              
    title="Colture predette",                                
    title_fontsize=11,                                       
    fontsize=9,                                              
    frameon=True,                                            
    facecolor="white",                                       
    framealpha=0.95                                          
)                                                            
                                                                
plt.tight_layout()                                           
plt.show()